<a href="https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### My ranked actions

My lane is content refresh. I will rank pages that may need review and give a simple reason for each recommendation.

Pages with higher priority should be reviewed first. The reason codes help the content team understand why a page was selected.

Example reason codes:

- STALE_VISIBLE: The page has not been updated for a long time but still receives impressions.
- DECLINING: The page shows a downward trend.
- LOW_CTR: The page gets impressions but has a relatively low click-through rate.
- OLD_CONTENT: The content is relatively old and may need review.

The ranking is a decision-support tool. It does not automatically mean that a page must be changed.

In [3]:
import os
import sys
import subprocess
import pandas as pd

REPO_URL = "https://github.com/shweta-1202/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

# Clone your GitHub repository
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# Go into the repository
os.chdir(REPO_DIR)

print("Current folder:", os.getcwd())

# Check that the data exists
data_path = "data/raw/content_refresh_anonymized.csv"

if not os.path.exists(data_path):
    print("Data file not found.")
    print("Files available in data/raw:")
    print(os.listdir("data/raw") if os.path.exists("data/raw") else "data/raw folder does not exist")
else:
    print("Data file found!")

# Load data
df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Current folder: /content/flyrank-ml-internship
Data file found!
Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [5]:
import pandas as pd
import numpy as np
import os

# Load the starter data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create simple action scores
df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["declining"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

# Action score
df["action_score"] = (
    df["stale"] * df["visible"] * df["impressions_90d"]
    + df["declining"] * df["impressions_90d"]
)

# Reason code
def reason(row):
    if row["stale"] == 1 and row["visible"] == 1:
        return "STALE_VISIBLE"
    elif row["declining"] == 1:
        return "DECLINING"
    elif row["visible"] == 1:
        return "VISIBLE"
    else:
        return "REVIEW"

df["reason_code"] = df.apply(reason, axis=1)

# Rank pages
queue = df.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

queue["priority"] = range(1, len(queue) + 1)

# Show top 20
top20 = queue.head(20)

display(
    top20[
        [
            "priority",
            "action_score",
            "reason_code",
            "impressions_90d",
            "days_since_last_update",
            "trend_direction"
        ]
    ]
)

,priority,action_score,reason_code,impressions_90d,days_since_last_update,trend_direction
0,1,517715,DECLINING,517715,104,down
1,2,509252,DECLINING,509252,20,down
2,3,463103,DECLINING,463103,20,down
3,4,416180,DECLINING,416180,22,down
4,5,347399,DECLINING,347399,104,down
5,6,309910,DECLINING,309910,104,down
6,7,309192,DECLINING,309192,104,down
7,8,236803,DECLINING,236803,20,down
8,9,233561,DECLINING,233561,104,down
9,10,228566,DECLINING,228566,20,down


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

This playbook is intended for a content team to decide which pages should be reviewed first.

The output helps prioritize human review. It can identify pages that look stale, visible, or declining based on the available data.

The recommendations are not automatic instructions to edit a page.

The model cannot tell us exactly why Google changed a ranking, and it cannot prove that updating a page will increase traffic.

A content specialist should review the page before making any change.

In [6]:
print("Intended user: Content team")
print("Purpose: Prioritize pages for human review")
print("Limit: Recommendations are decision-support, not automatic actions.")

Intended user: Content team
Purpose: Prioritize pages for human review
Limit: Recommendations are decision-support, not automatic actions.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review

Before changing a page, a person should check:

1. Whether the content is still accurate.
2. Whether the page satisfies the user's search intent.
3. Whether the information is outdated.
4. Whether important information is missing.
5. Whether the title and description are appropriate.
6. Whether the recommendation makes sense for the actual page.

### No-go list

The system should never automatically:

- Delete a page.
- Rewrite an entire article without human review.
- Change important factual information automatically.
- Make claims about Google's algorithm.
- Guarantee that an update will increase traffic.
- Publish changes directly to a website.

In [7]:
print("Human review is required before any content change.")
print("No automatic publishing or deletion is allowed.")

Human review is required before any content change.
No automatic publishing or deletion is allowed.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retrain triggers

The recommendations should be monitored over time.

I would review or retrain the model when:

- The data distribution changes significantly.
- The model's Precision@K decreases.
- The relationship between the features and declining pages changes.
- New types of content appear.
- The content or search environment changes substantially.
- The recommendations are repeatedly judged incorrect by human reviewers.

The model should not be considered permanently correct. Its performance should be checked regularly.

In [9]:
print("Monitoring checks:")
print("1. Model Precision@K")
print("2. Feature/data distribution")
print("3. Human review feedback")
print("4. Changes in content types")

Monitoring checks:
1. Model Precision@K
2. Feature/data distribution
3. Human review feedback
4. Changes in content types


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [10]:
import os

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Save the ranked action queue
output_file = "work/outputs/content_action_playbook.csv"

queue.head(100).to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)

# Show saved results
display(queue.head(20))

Saved: work/outputs/content_action_playbook.csv


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,impression_tier,position_tier,trend_direction,trend_pct,stale,visible,declining,action_score,reason_code,priority
0,content_5fe46e04994d,client_4e07408562,1900.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,excellent,page_1,down,-44.8,0,1,1,517715,DECLINING,1
1,content_8c19996aa890,client_4e07408562,70.0,0.01,LOW,0.00,keyword article,informational,2895.0,19343.0,...,excellent,top_3,down,-44.5,0,1,1,509252,DECLINING,2
2,content_4c36c775b818,client_4e07408562,40.0,0.00,LOW,0.00,keyword article,informational,3097.0,20514.0,...,excellent,top_3,down,-33.2,0,1,1,463103,DECLINING,3
3,content_1a9e894be2e2,client_19581e27de,70.0,0.07,LOW,0.10,keyword article,transactional,NaN,NaN,...,excellent,page_1,down,-27.0,0,1,1,416180,DECLINING,4
4,content_2c2606c5d176,client_19581e27de,590.0,0.18,LOW,0.31,keyword article,commercial,NaN,NaN,...,excellent,page_1,down,-36.5,0,1,1,347399,DECLINING,5
5,content_cb112fce36be,client_19581e27de,70.0,0.65,MEDIUM,0.34,keyword article,transactional,2761.0,18472.0,...,excellent,page_1,down,-41.8,0,1,1,309910,DECLINING,6
6,content_9532f197bbc8,client_4e07408562,10.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,excellent,top_3,down,-37.3,0,1,1,309192,DECLINING,7
7,content_008fb02c46cb,client_349c41201b,10.0,0.86,HIGH,0.00,keyword article,commercial,3338.0,21685.0,...,excellent,page_1,down,-21.3,0,1,1,236803,DECLINING,8
8,content_813e88069237,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,commercial,4610.0,30146.0,...,excellent,page_3_5,down,-33.8,0,1,1,233561,DECLINING,9
9,content_ff94c9b6b411,client_349c41201b,20.0,0.00,LOW,0.00,keyword article,informational,5375.0,35334.0,...,excellent,page_3_5,down,-32.4,0,1,1,228566,DECLINING,10


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.